In [ ]:
import os
import sys
import shutil
import cv2
import matplotlib.pyplot as plt
import numpy as np
import shutil

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/video_surgery')

from src import config, setup
setup.download_davis()

In [ ]:
source_frames = sorted(config.SEQ_DIR.glob('*.jpg'))
native_fps    = 24
target_fps    = config.specs["fps"]
interval      = max(1, round(native_fps / target_fps))

print(f"Source frames: {len(source_frames)}")
print(f"Sampling every {interval} frames ({native_fps}→{target_fps} FPS)")

saved = []
for i, src in enumerate(source_frames):
    if i % interval == 0:
        dst = config.FRAMES / f"{len(saved)}.jpg"
        if not dst.exists():
            shutil.copy(src, dst)
        saved.append(dst)

print(f"Saved {len(saved)} frames to {config.FRAMES}")

In [ ]:
GT_OUT = config.MASKS / 'gt'
GT_OUT.mkdir(exist_ok=True)

In [ ]:
gt_frames = sorted(config.GT_DIR.glob('*.png'))
saved_gt = []

for i, src in enumerate(gt_frames):
    if i % interval == 0:
        dst = GT_OUT / f"{len(saved_gt)}.png"
        if not os.path.exists(dst):
            shutil.copy(src, dst)
        saved_gt.append(dst)

print(f"Saved {len(saved_gt)} GT masks to {GT_OUT}")

In [ ]:
frame  = cv2.imread(config.FRAMES / '0.jpg')
frame  = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
gt     = cv2.imread(GT_OUT / '0.png', cv2.IMREAD_GRAYSCALE)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(frame)
axes[0].set_title('Frame 0: original')
axes[0].axis('off')

overlay = frame.copy()
overlay[gt > 0] = (overlay[gt > 0] * 0.5 + np.array([255, 0, 0]) * 0.5).astype('uint8')
axes[1].imshow(overlay)
axes[1].set_title('Frame 0: GT mask overlay')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f"Frame shape:   {frame.shape}")
print(f"GT mask shape: {gt.shape}")